# FlowAI: Qwen2.5-0.5B LoRA Fine-Tuning in JupyterLab

This notebook fine-tunes **Qwen/Qwen2.5-0.5B-Instruct** using **LoRA (Low-Rank Adaptation)** on your approved FlowAI dictations.

- **Task:** Converts raw speech transcripts with verbal fillers (*um*, *uh*, *like*, *you know*) into polished `Professional`, `Casual`, or `Simple` text.
- **Hardware:** Works on a free GPU (Google Colab T4, Kaggle, Paperspace, or local Nvidia GPU).


## Step 1: Install Dependencies
Checks for PyTorch (already installed in Google Colab) and installs transformers, peft, datasets, and accelerate with timeout protection.

In [ ]:
import sys

# 1. Check or install PyTorch
try:
    import torch
    print(f'✓ PyTorch already installed: v{torch.__version__}')
except ImportError:
    print('Installing PyTorch...')
    !{sys.executable} -m pip install --default-timeout=1000 torch --extra-index-url https://download.pytorch.org/whl/cpu

# 2. Install Hugging Face training ecosystem
print('Installing transformers, peft, accelerate, datasets...')
!{sys.executable} -m pip install --default-timeout=1000 transformers peft accelerate datasets

# 3. Optional bitsandbytes (Linux/Colab GPU only)
try:
    import bitsandbytes
    print('✓ bitsandbytes available')
except Exception:
    try:
        !{sys.executable} -m pip install bitsandbytes
    except Exception:
        print('Note: bitsandbytes skipped (not required for standard training).')

## Step 2: Check GPU & Environment
Verify compute device (CUDA GPU or CPU) and available VRAM.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using compute device: {device.upper()}')
if device == 'cuda':
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'Available VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('Running on CPU. (Tip: For 10x faster training, open this notebook in Google Colab with free T4 GPU!)')

## Step 3: Load the FlowAI Dataset
Loads `dataset.jsonl` exported from FlowAI's History page (or uses built-in demo examples if not uploaded yet).

In [ ]:
import os
import json
from datasets import Dataset

DATASET_PATH = 'dataset.jsonl'

if os.path.exists(DATASET_PATH):
    records = []
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    raw_dataset = Dataset.from_list(records)
    print(f'Loaded {len(raw_dataset)} approved examples from {DATASET_PATH}')
else:
    print('dataset.jsonl not found. Using fallback demo dataset...')
    demo_samples = [
        {'style': 'Professional', 'raw': 'um please send the financial report by like tomorrow', 'cleaned': 'Please send the financial report by tomorrow.'},
        {'style': 'Casual', 'raw': 'hey um you know let us meet at 3 no wait 4pm', 'cleaned': 'Hey, let\'s meet at 4:00 PM.'},
        {'style': 'Simple', 'raw': 'so basically the server is down and we need to restart it', 'cleaned': 'The server is down. Please restart it.'},
        {'style': 'Professional', 'raw': 'sun yaar we need to approve this pull request na', 'cleaned': 'Please review and approve this pull request at your earliest convenience.'}
    ] * 20
    raw_dataset = Dataset.from_list(demo_samples)
    print(f'Loaded {len(raw_dataset)} demo examples.')

print('Sample row:', raw_dataset[0])

## Step 4: Load Base Model & Tokenizer
We load `Qwen/Qwen2.5-0.5B-Instruct` (~490M parameters).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Loading tokenizer and model: {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None,
    trust_remote_code=True,
)
print('Model loaded successfully!')

## Step 5: Attach LoRA (Low-Rank Adaptation)
Freeze the base model and add lightweight trainable adapters ($r=16, \alpha=32$ on $q, k, v, o$ projections).

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6: Preprocess & Mask Prompt Tokens (-100)
Mask prompt tokens with label `-100` so loss is only calculated on the target cleaned text.

In [ ]:
def build_prompt(style, raw):
    sys_prompt = f'You are FlowAI\'s intelligent dictation cleanup engine. Rewrite into polished {style} text. Remove fillers (um, uh, like), fix grammar. Output ONLY the cleaned text.'
    messages = [
        {'role': 'system', 'content': sys_prompt},
        {'role': 'user', 'content': f'Raw speech: {raw}'},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def preprocess_function(batch):
    input_ids_list, labels_list = [], []
    for style, raw, cleaned in zip(batch['style'], batch['raw'], batch['cleaned']):
        prompt_text = build_prompt(style, raw)
        prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
        response_ids = tokenizer.encode(f'{cleaned}{tokenizer.eos_token}', add_special_tokens=False)
        input_ids = prompt_ids + response_ids
        labels = [-100] * len(prompt_ids) + response_ids
        input_ids_list.append(input_ids)
        labels_list.append(labels)
    return {'input_ids': input_ids_list, 'labels': labels_list}

tokenized = raw_dataset.map(preprocess_function, batched=True, remove_columns=raw_dataset.column_names)
split = tokenized.train_test_split(test_size=0.15, seed=42)
print(f'Train size: {len(split["train"])}, Validation size: {len(split["test"])}')

## Step 7: Train with Hugging Face Trainer
Runs fine-tuning for 3 epochs.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir='./flowai-lora-adapter',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_ratio=0.1,
    logging_steps=5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors='pt'),
)

trainer.train()

## Step 8: Test Live Predictions in the Notebook

In [ ]:
model.eval()

def test_cleanup(raw_text, style='Professional'):
    prompt = build_prompt(style, raw_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.1, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print('Test 1 (Professional):', test_cleanup('um we should like review the code tomorrow morning', 'Professional'))
print('Test 2 (Casual):', test_cleanup('sun yaar let us meet at 4pm na', 'Casual'))
print('Test 3 (Simple):', test_cleanup('basically the payment service failed five hundred error', 'Simple'))

## Step 9: Save & Merge the Model
Merges the LoRA adapter back into the base model so it can be served standalone via FastAPI (`serve_local.py`).

In [ ]:
from peft import PeftModel

ADAPTER_DIR = './flowai-lora-adapter'
MERGED_DIR = './flowai-qwen-merged'

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to {ADAPTER_DIR}')

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32, device_map='cpu')
merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR).merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f'Standalone merged model saved to {MERGED_DIR}!')